# Agentic Artificial Intelligence
## Exercise - Unit 06: Model Context Protocol

Welcome to the sixth unit of the Agentic Artificial Intelligence course!

## Learning Objectives
By the end of this lesson, students will:
1. Understand what the Model Context Protocol (MCP) is and why it's important
2. Learn the difference between custom tools and MCP servers
3. Understand how MCP enables standardized integration with external services
4. Learn how to obtain and configure MCP servers from external providers
5. Understand how to integrate MCP tools into LangChain/LangGraph agents
6. Create agents that leverage MCP servers for real-time data access
7. Understand security considerations when using MCP servers

## Prerequisites
- Students should have completed Unit 05 exercises on tool integration
- Understanding of LangGraph basics (Unit 03)
- Familiarity with the `BaseAgent`, `SimpleAgent`, and `ToolAgent` classes
- Basic Python knowledge (classes, inheritance, abstract methods)
- Understanding of API keys and environment variables

## 1. What is the Model Context Protocol (MCP)?

The **Model Context Protocol (MCP)** is an open standard developed by Anthropic that enables AI systems to securely and efficiently connect with external data sources, tools, and services. Think of MCP as a standardized "bridge" that allows AI agents to access real-time information and capabilities beyond their training data.

### Why Do We Need MCP?

In Unit 05, you learned how to create custom tools. While custom tools work great for your own code, MCP provides several advantages:

1. **Standardized Integration** - MCP servers follow a common protocol, making it easy to integrate multiple services
2. **External Services** - Access powerful services (web search, databases, APIs) without building everything yourself
3. **Security** - MCP provides secure, controlled access to external resources
4. **Ecosystem** - Many providers offer MCP servers, giving you access to a wide range of capabilities
5. **Separation of Concerns** - MCP servers run independently, keeping your agent code clean

### How MCP Works

```
┌─────────────┐         MCP Protocol         ┌──────────────┐
│             │ ◄──────────────────────────► │              │
│ AI Agent    │                              │ MCP Server   │
│ (Your Code) │                              │ (External)   │
│             │                              │              │
└─────────────┘                              └──────────────┘
     │                                              │
     │                                              │
     └─── Uses MCP Tools ───────────────────────────┘
```

**Key Components:**
- **MCP Server**: A service that provides tools/resources (e.g., Tavily web search)
- **MCP Client**: Your agent that connects to and uses MCP servers
- **MCP Protocol**: The standardized communication layer between them

### MCP vs Custom Tools

| Aspect | Custom Tools | MCP Servers |
|--------|-------------|-------------|
| **Location** | In your codebase | External service |
| **Setup** | Write code yourself | Connect to existing server |
| **Maintenance** | You maintain it | Provider maintains it |
| **Capabilities** | Limited to your code | Access to powerful external services |
| **Use Case** | Simple, specific needs | Complex, external data/services |

# Update dependencies

As I added new packages, you must first run `uv sync`in order to run the code.

In [1]:
!uv sync

Resolved 236 packages in 0.95ms
Audited 206 packages in 0.16ms


If you face import issues run `uv sync` in the terminal from the root dir of this project.

## 4. Integrate MCPs from external providers

In [1]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "langgraph-docs-mcp": {
            "command": "uvx",
                "args": [
                    "--from",
                    "mcpdoc",
                    "mcpdoc",
                    "--urls",
                    "LangGraph:https://langchain-ai.github.io/langgraph/llms.txt LangChain:https://python.langchain.com/llms.txt",
                ],
                "transport": "stdio"
        }
    }
)

tools = await client.get_tools()

for i, tool in enumerate(tools, 1):
    print(f"{i}. {tool.name}")
    if tool.description:
        # Truncate long descriptions
        desc = tool.description
        print(f"   {desc}")
        
    print()

1. list_doc_sources
   List all available documentation sources.

        This is the first tool you should call in the documentation workflow.
        It provides URLs to llms.txt files or local file paths that the user has made available.

        Returns:
            A string containing a formatted list of documentation sources with their URLs or file paths
        

2. fetch_docs
   Fetch and parse documentation from a given URL or local file.

Use this tool after list_doc_sources to:
1. First fetch the llms.txt file from a documentation source
2. Analyze the URLs listed in the llms.txt file
3. Then fetch specific documentation pages relevant to the user's question

Args:
    url: The URL to fetch documentation from.

Returns:
    The fetched documentation content converted to markdown, or an error message
    if the request fails or the URL is not from an allowed domain.



In [2]:
# emulate tool call with 
print(f'Calling tool {tools[0].name}')
response = await tools[0].ainvoke({'tool_name': 'list_doc_sources'})
print(response)

print(f'Calling tool {tools[1].name}')
response = await tools[1].ainvoke({'url': 'https://langchain-ai.github.io/langgraph/concepts/server-mcp/'})
print(response)

Calling tool list_doc_sources
LangGraph
URL: https://langchain-ai.github.io/langgraph/llms.txt LangChain:https://python.langchain.com/llms.txt


Calling tool fetch_docs
Redirecting...


Redirecting...


We see we get the message "Redirecting" from the call of the second tool. This is not our mistake but the tool although provided by langchain (one of the agentic AI leaders) does not work. I came up with a workaround ;)

In [3]:
from agentic_ai.tools.url_fetcher import UrlFetcherTool

response = UrlFetcherTool().to_langchain_tool().invoke({'url': 'https://langchain-ai.github.io/langgraph/concepts/server-mcp/', 'extract_text': True})
print(f"Response: {response[:1000]}...")

Response: Title: MCP endpoint in Agent Server - Docs by LangChain

The Model Context Protocol (MCP) is an open protocol for describing tools and data sources in a model-agnostic format, enabling LLMs to discover and use them via a structured API.
Agent Server implements MCP using the Streamable HTTP transport. This allows LangGraph agents to be exposed as MCP tools, making them usable with any MCP-compliant client supporting Streamable HTTP.
The MCP endpoint is available at /mcp on Agent Server.
You can set up custom authentication middleware to authenticate a user with an MCP server to get access to user-scoped tools within your LangSmith deployment.
An example architecture for this flow:
​Requirements
To use MCP, ensure you have the following dependencies installed:
langgraph-api >= 0.2.3
langgraph-sdk >= 0.1.61
Install them with:
pipuvCopypip install "langgraph-api>=0.2.3" "langgraph-sdk>=0.1.61"
​Usage overview
To enable MCP:
Upgrade to use langgraph-api>=0.2.3. If you are deployin

## 5. Creating Your Own MCP Server

Now let's create our own MCP server that exposes the `UrlFetcherTool` we created earlier. This demonstrates how you can wrap your custom tools as MCP servers.

### 5.1 Understanding the UrlFetcherTool

First, let's examine the `UrlFetcherTool` that we'll expose as an MCP tool:

In [4]:
from agentic_ai.tools.url_fetcher import UrlFetcherTool

# Show the UrlFetcherTool
print("UrlFetcherTool Information:")
print("=" * 60)
fetcher = UrlFetcherTool()
print(f"Name: {fetcher.name}")
print(f"\nDescription:\n{fetcher.description}")

UrlFetcherTool Information:
Name: fetch_url

Description:

            Fetch content from a URL using HTTP requests. This tool AUTOMATICALLY handles redirects
            and returns the actual content from the final destination URL.
            
            CRITICAL: Use this tool instead of fetch_docs when:
            - fetch_docs returns "Redirecting..." 
            - You need to fetch specific documentation pages
            - You encounter any redirect-related issues
            
            This tool follows HTTP redirects automatically and extracts readable text from HTML pages.
            It is the preferred tool for fetching LangGraph and LangChain documentation pages.
            


### 5.2 Creating an MCP Server

We've created an MCP server in `agentic_ai/mcp/url_fetcher_server.py` that exposes the `UrlFetcherTool` as an MCP tool. Let's examine it:


In [5]:
# Show the MCP server code
import inspect
from agentic_ai.mcp import url_fetcher_server

print("MCP Server Code:")
print("=" * 60)
print(inspect.getsource(url_fetcher_server))


MCP Server Code:
"""
MCP Server that exposes the UrlFetcherTool as an MCP tool.

This server can be run as a standalone MCP server using stdio transport.
"""

import asyncio
import sys
from typing import Any

from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import Tool, TextContent

# Import the UrlFetcherTool from the parent package
from agentic_ai.tools.url_fetcher import UrlFetcherTool


# Create the server instance
server = Server("url-fetcher-mcp-server")

# Create the tool instance
url_fetcher = UrlFetcherTool()


@server.list_tools()
async def list_tools() -> list[Tool]:
    """List available tools."""
    return [
        Tool(
            name="fetch_url",
            description=url_fetcher.description,
            inputSchema={
                "type": "object",
                "properties": {
                    "url": {
                        "type": "string",
                        "description": "The URL to fetch (must be a valid HT

### 5.3 Using the MCP Server

Now let's connect to our custom MCP server and use it with an agent. The server can be run as a subprocess using Python:


In [4]:
import sys
import os

# Get the path to the MCP server script
mcp_server_path = os.path.join(
    os.path.dirname(os.path.dirname(os.path.dirname(os.getcwd()))),
    "agentic_ai", "mcp", "url_fetcher_server.py"
)

# Alternative: use Python module path
python_path = sys.executable

print(f"Python executable: {python_path}")
print(f"MCP server path: {mcp_server_path}")
print(f"\nTo run the MCP server, use:")
print(f"  {python_path} -m agentic_ai.mcp.url_fetcher_server")


Python executable: /Users/tockenga/Programming/agentic_artificial_intelligence-1/.venv/bin/python3
MCP server path: /Users/tockenga/Programming/agentic_ai/mcp/url_fetcher_server.py

To run the MCP server, use:
  /Users/tockenga/Programming/agentic_artificial_intelligence-1/.venv/bin/python3 -m agentic_ai.mcp.url_fetcher_server


### 5.4 Alternative method to implement mcp (FastMCP)

There is another method to implelment mcp. Checkout `arxiv.py` where I used FastMCP. I will not go into detail but also use this mcp in the follwing.

In [5]:
from agentic_ai.utils.paths import DATA_DIR

ARXIV_DATA_DIR = (DATA_DIR / "arxiv").as_posix()

In [9]:
# Connect to our custom MCP server
from langchain_mcp_adapters.client import MultiServerMCPClient

# Create client for our url-fetcher MCP server
mcp_client = MultiServerMCPClient(
    {
        "langgraph-docs-mcp": {
            "command": "uvx",
                "args": [
                    "--from",
                    "mcpdoc",
                    "mcpdoc",
                    "--urls",
                    "LangGraph:https://langchain-ai.github.io/langgraph/llms.txt LangChain:https://python.langchain.com/llms.txt",
                ],
                "transport": "stdio"
        },
        "url-fetcher-mcp": {
            "command": sys.executable,
            "args": [
                "-m",
                "agentic_ai.mcp.url_fetcher_server"
            ],
            "transport": "stdio"
        },
        "arxiv-server": {
            "command": sys.executable,
            "args": [
                "-m",
                "agentic_ai.mcp.arxiv"
            ],
            "env": {
                "DOWNLOAD_PATH": ARXIV_DATA_DIR
            },
            "transport": "stdio"
        }
    }
)

# Get tools from our MCP server
tools = await mcp_client.get_tools()

print(f"✅ Successfully connected to URL Fetcher MCP server!")
print(f"Found {len(tools)} tool(s):\n")

for i, tool in enumerate(tools, 1):
    print(f"{i}. {tool.name}")
    if tool.description:
        desc = tool.description[:200] + "..." if len(tool.description) > 200 else tool.description
        print(f"   {desc}\n")
        print(f'    {tool.args}')
    print()


✅ Successfully connected to URL Fetcher MCP server!
Found 8 tool(s):

1. list_doc_sources
   List all available documentation sources.

        This is the first tool you should call in the documentation workflow.
        It provides URLs to llms.txt files or local file paths that the user ha...

    {}

2. fetch_docs
   Fetch and parse documentation from a given URL or local file.

Use this tool after list_doc_sources to:
1. First fetch the llms.txt file from a documentation source
2. Analyze the URLs listed in the l...

    {'url': {'title': 'Url', 'type': 'string'}}

3. fetch_url
   
            Fetch content from a URL using HTTP requests. This tool AUTOMATICALLY handles redirects
            and returns the actual content from the final destination URL.
            
           ...

    {'url': {'type': 'string', 'description': 'The URL to fetch (must be a valid HTTP/HTTPS URL)'}, 'extract_text': {'type': 'boolean', 'description': 'Whether to extract text from HTML (default: True).

#### Testing some of the tools

In [10]:
# use the arxiv search tool
articles = await tools[7].ainvoke({'title': 'model context protocol'})

import json
# convert result to dict to display nicely
articles = json.loads(articles)
for index, title in enumerate(articles):
    print(f"{index}. {title}")

0. Exploiting Context to Identify Lexical Atoms -- A Statistical View of Linguistic Context
1. Adaptation of TURN protocol to SIP protocol
2. MCP4EDA: LLM-Powered Model Context Protocol RTL-to-GDSII Automation with Backend Aware Synthesis Optimization
3. An Ontology-Based Reasoning Framework for Context-Aware Applications
4. Kak's three-stage protocol of secure quantum communication revisited: Hitherto unknown strengths and weaknesses of the protocol
5. Pingmark: A Textual Protocol for Universal Spatial Mentions
6. Node Disjoint Multipath Routing Considering Link and Node Stability protocol: A characteristic Evaluation
7. Conversion of a general quantum stabilizer code to an entanglement distillation protocol
8. A Protocol for KG Construction Tasks Involving Users
9. A simple quantum oblivious transfer protocol


In [11]:
# Test fetch_url tool to retrieve langchain docs
print("Testing the fetch_url tool from MCP server:")
print("=" * 60)

# Test with a simple URL
test_url = "https://langchain-ai.github.io/langgraph/concepts/server-mcp/"
result = await url_fetcher_tools[2].ainvoke({
    "url": test_url,
    "extract_text": True
})

print(f"Fetched content (first 500 chars):\n{result[:500]}...")


Testing the fetch_url tool from MCP server:
Fetched content (first 500 chars):
Title: MCP endpoint in Agent Server - Docs by LangChain

The Model Context Protocol (MCP) is an open protocol for describing tools and data sources in a model-agnostic format, enabling LLMs to discover and use them via a structured API.
Agent Server implements MCP using the Streamable HTTP transport. This allows LangGraph agents to be exposed as MCP tools, making them usable with any MCP-compliant client supporting Streamable HTTP.
The MCP endpoint is available at /mcp on Agent Server.
You can s...


## 6. Integrating MCP Tools into an Agent

Now let's integrate both MCP servers (the docs MCP and our custom URL fetcher MCP) into an agent:


In [25]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent

# Configure Gemini model for tool calling
model = init_chat_model("google_genai:gemini-2.0-flash-lite")
memory = InMemorySaver()

async def build_agent_with_mcp():
    """Build an agent with both MCP servers."""
    # Get tools from both MCP servers
    tools = await mcp_client.get_tools()  # our custom url-fetcher-mcp
    
    # Create the agent
    agent = create_react_agent(model, tools, checkpointer=memory)
    return agent

async def list_all_tools():
    """List all available tools from MCP servers."""
    tools = await mcp_client.get_tools()
    
    print("Available MCP Tools:")
    print("=" * 60)
    for t in tools:
        print(f"  - {t.name}: {t.description[:80]}...")

# Build the agent
agent = await build_agent_with_mcp()
config = {"configurable": {"thread_id": "mcp_demo_123"}}

# List all tools
await list_all_tools()

I0000 00:00:1763547353.260341 10607641 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1763547353.269374 10607641 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1763547353.274612 10607641 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
/var/folders/fk/_r3hvp4d0qx6w_ny6brbf6nw0000gn/T/ipykernel_34435/4126337589.py:15: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model, tools, checkpointer=memory)
I0000 00:00:1763547355.474839 10607641 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1763547355.479866 10607641 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handl

Available MCP Tools:
  - list_doc_sources: List all available documentation sources.

        This is the first tool you sh...
  - fetch_docs: Fetch and parse documentation from a given URL or local file.

Use this tool aft...
  - fetch_url: 
            Fetch content from a URL using HTTP requests. This tool AUTOMATICAL...
  - get_article_url: 
Retrieve the URL of an article hosted on arXiv.org based on its title. Use this...
  - download_article: 
Download the article hosted on arXiv.org as a PDF file. This tool can search fo...
  - load_article_to_context: 
Load the article hosted on arXiv.org into context. This tool searches for the a...
  - get_details: 
Retrieve information of an article hosted on arXiv.org based on its title. This...
  - search_arxiv: 
Performs a search query on the arXiv API based on specified parameters and retu...


In [26]:
# Helper functions for interacting with the agent
async def chat(agent, message):
    print(f"\nQuery: {message}")
    print("=" * 60)
    
    result = await agent.ainvoke({
        "messages": [
            {"role": "user", "content": message}
        ]
    }, config)
    
    # Print the last message
    if result["messages"]:
        result["messages"][-1].pretty_print()

def get_full_conversation(agent):
    """Get full conversation history."""
    snapshot = agent.get_state(config)
    if snapshot and snapshot.values:
        messages = snapshot.values.get("messages", [])
        print(f"Total messages: {len(messages)}\n")
        for i, msg in enumerate(messages, 1):
            print(f"{i}. {msg.__class__.__name__}:")
            print(f"   {msg.content}\n")
    else:
        print("No messages found in memory for this thread_id")


In [27]:
# Test the agent with a query that requires URL fetching
test_query = "Tell me about langgraph"

await chat(agent, test_query)


Query: Tell me about langgraph


I0000 00:00:1763547366.914506 10607641 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1763547368.108488 10607641 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


================================== Ai Message ==================================

LangGraph is a framework for building stateful, agentic applications with LLMs. It allows developers to create reliable, extensible, and streaming-enabled AI agents. Here's a summary based on the documentation:

*   **Key Features:**
    *   Building agentic systems.
    *   Reliability, extensibility, and streaming support.
    *   State management and human-in-the-loop controls.
*   **Tutorials:**
    *   Building a basic chatbot.
    *   Integrating web search tools.
    *   Implementing memory in chatbots.
    *   Human-in-the-loop controls.
    *   Customizing state.
    *   Implementing time travel.
*   **Concepts:**
    *   Agent architectures.
    *   Workflows and agents.
    *   Core concepts and components (States, Nodes, Edges).
    *   Streaming capabilities.
    *   Persistence and Checkpointing.
    *   Human-in-the-loop workflows.
    *   Tools.
    *   Subgraphs.
    *   Multi-agent syste

In [28]:
get_full_conversation(agent)

Total messages: 6

1. HumanMessage:
   Tell me about langgraph

2. AIMessage:
   

3. ToolMessage:
   LangGraph
URL: https://langchain-ai.github.io/langgraph/llms.txt LangChain:https://python.langchain.com/llms.txt



4. AIMessage:
   I will start by fetching the documentation from the LangGraph URL.

5. ToolMessage:
   # Guides
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/index/): This page provides an overview of the LangGraph project, including its logo and essential scripts for functionality within MkDocs. It also includes a reference to the README.md file for detailed information about the project. The content is designed to be user-friendly and visually appealing.
- [LangGraph Quickstart Guide](https://langchain-ai.github.io/langgraph/agents/agents/): This quickstart guide provides step-by-step instructions for setting up and using LangGraph's prebuilt components to create agentic systems. It covers prerequisites, installation, agent creation, configuratio

In [31]:
# Test the agent with a query that requires URL fetching
test_query = "Is there an article about langgraph?"

await chat(agent, test_query)


Query: Is there an article about langgraph?


I0000 00:00:1763547462.580085 10607641 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


================================== Ai Message ==================================

Yes, there are multiple articles on arXiv that mention "LangGraph". Here are a few, along with their titles and authors:

*   "Agent AI with LangGraph: A Modular Framework for Enhancing Machine Translation Using Large Language Models" by Jialin Wang, Zhihua Duan
*   "Empirical Research on Utilizing LLM-based Agents for Automated Bug Fixing via LangGraph" by Jialin Wang, Zhihua Duan
*   "Exploration of LLM Multi-Agent Application Implementation Based on LangGraph+CrewAI" by Zhihua Duan, Jialin Wang
*   "Intelligent Spark Agents: A Modular LangGraph Framework for Scalable, Visualized, and Enhanced Big Data Machine Learning Workflows" by Jialin Wang, Zhihua Duan


In [34]:
# Test the agent with a query that requires URL fetching
test_query = "Please download the first one you mentioned."

await chat(agent, test_query)


Query: Please download the first one you mentioned.
================================== Ai Message ==================================

I am sorry, I am unable to download the article. It seems there was an issue retrieving the article. This might be due to an incorrect or incomplete title, or because the work has not been published on arXiv.


In [37]:
# Test the agent with a query that requires URL fetching
test_query = "Please try again with the title: Agent AI with LangGraph: A Modular Framework for Enhancing Machine Translation Using Large Language Models."

await chat(agent, test_query)


Query: Please try again with the title: Agent AI with LangGraph: A Modular Framework for Enhancing Machine Translation Using Large Language Models.


I0000 00:00:1763547676.578238 10607641 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


================================== Ai Message ==================================

I am sorry, I am unable to download the article. It seems there was an issue retrieving the article. This might be due to an incorrect or incomplete title, or because the work has not been published on arXiv.


As we can see not everything works seemless right away. At this point you need to experiment with different prompts, architectures etc..

However, we created our first agent that uses multiple mcp servers, both, externally and "internally" hosted ones, to address different tasks.

## 7. Understanding MCP vs Direct API Integration

Let's compare the two approaches we've seen:

### Approach 1: Direct API Integration (What we just did)
- **Pros:**
  - Simple and straightforward
  - Direct control over API calls
  - Easy to debug
  - Works well for single services
  
- **Cons:**
  - Need to write custom code for each service
  - Not standardized across providers
  - More maintenance overhead

### Approach 2: MCP Protocol (Standardized)
- **Pros:**
  - Standardized protocol across all MCP servers
  - Easy to switch between providers
  - Consistent interface
  - Better for production systems
  - Supports multiple services through one protocol
  
- **Cons:**
  - Requires MCP client setup
  - Slightly more complex initial setup
  - Need to understand MCP protocol

### When to Use Each?

- **Use Direct API** when:
  - You're prototyping quickly
  - You only need one or two services
  - You want maximum control
  
- **Use MCP** when:
  - You need multiple external services
  - You want standardized integration
  - You're building production systems
  - You want to leverage the MCP ecosystem


## 8. Security Best Practices

When working with MCP servers and external APIs, security is crucial:

### 1. **Never Hardcode API Keys**
```python
# ❌ BAD - Never do this!
api_key = "tvly-1234567890abcdef"

# ✅ GOOD - Use environment variables
api_key = os.getenv("TAVILY_API_KEY")
```

### 2. **Use .env Files**
- Create a `.env` file in your project root
- Add it to `.gitignore` to prevent committing secrets
- Load it with `python-dotenv`

### 3. **Rotate Keys Regularly**
- Change API keys periodically
- Revoke old keys when generating new ones

### 4. **Limit Key Permissions**
- Use the minimum permissions needed
- Some services allow you to restrict what keys can do

### 5. **Monitor Usage**
- Check your API usage regularly
- Set up alerts for unusual activity
- This helps detect if your key is compromised

### 6. **Use Different Keys for Different Environments**
- Development keys for testing
- Production keys for live systems
- Never mix them up!


## 9. Exercise

### Create a File System MCP Server

In this exercise, you will create your own MCP server that gives an AI agent access to your computer's file system. This is a powerful capability that allows agents to analyze codebases, read documentation, explore directory structures, and more.

#### Objectives

1. **Create a file system MCP server** that exposes tools for:
   - Listing directory contents
   - Reading file contents
   - Searching for files by name or pattern
   - Getting file metadata (size, modification time, etc.)

2. **Integrate the MCP server** with a LangGraph agent

3. **Test the agent** by asking it to analyze this project directory or any other directory on your PC

#### Step 1: Create the MCP Server

Create a new file `agentic_ai/mcp/filesystem_server.py` that implements a file system MCP server. You can use either approach:

- **Option A**: Use the standard `mcp.server.Server` (like `url_fetcher_server.py`)
- **Option B**: Use `FastMCP` (like `arxiv.py`)

**Suggested Tools to Implement:**

1. **`list_directory`** - List files and directories in a given path
   - Parameters: `path` (string, required)
   - Returns: List of files and directories with their types

2. **`read_file`** - Read the contents of a file
   - Parameters: `path` (string, required), `max_lines` (integer, optional) to limit output
   - Returns: File contents (optionally truncated)

3. **`search_files`** - Search for files by name pattern
   - Parameters: `directory` (string, required), `pattern` (string, required), `recursive` (boolean, default: True)
   - Returns: List of matching file paths

4. **`get_file_info`** - Get metadata about a file or directory
   - Parameters: `path` (string, required)
   - Returns: File size, modification time, type, etc.

**Security Considerations:**

- ⚠️ **IMPORTANT**: Add path validation to prevent access to sensitive directories (e.g., `/etc`, `/home`, system directories)
- Consider adding a `BASE_PATH` environment variable to restrict access to a specific directory
- Validate that paths don't contain `..` (directory traversal attacks)
- Handle errors gracefully (file not found, permission denied, etc.)

#### Step 2: Test Your MCP Server

1. Run your MCP server manually to ensure it starts correctly:
   ```bash
   python -m agentic_ai.mcp.filesystem_server
   ```

2. Create a test client to verify your tools work:
   ```python
   from langchain_mcp_adapters.client import MultiServerMCPClient
   import sys
   
   mcp_client = MultiServerMCPClient({
       "filesystem": {
           "command": sys.executable,
           "args": ["-m", "agentic_ai.mcp.filesystem_server"],
           "env": {
               "BASE_PATH": "/path/to/allowed/directory"  # Optional: restrict access
           },
           "transport": "stdio"
       }
   })
   
   tools = await mcp_client.get_tools()
   # Test your tools here
   ```

#### Step 3: Integrate with an Agent

Create an agent that uses your file system MCP server and test it with queries like:

- "Analyze the structure of this project. What are the main components?"
- "Find all Python files in the `agentic_ai` directory"
- "Read the README.md file and summarize it"
- "What tools are available in the `tools` directory?"
- "Find all files that contain the word 'agent' in their name"

#### Step 4: Advanced Challenge (Optional)

If you want to go further, add these additional capabilities:

- **`write_file`** - Write content to a file (with safety checks!)
- **`create_directory`** - Create a new directory
- **`find_in_files`** - Search for text content within files (grep-like functionality)
- **`get_directory_tree`** - Get a visual tree structure of a directory

#### Hints

- Use Python's `pathlib` module for path handling (it's cross-platform)
- Use `os.walk()` or `pathlib.rglob()` for recursive directory traversal
- For reading files, handle encoding issues (try UTF-8 first, fallback to errors='ignore')
- Consider file size limits when reading files (don't try to read huge binary files)
- Use `os.path.getsize()`, `os.path.getmtime()` for file metadata

#### Example Test Query

Once your agent is set up, try this:

```python
# Build agent with filesystem MCP
agent = await build_agent_with_filesystem_mcp()

# Test query
response = await agent.ainvoke({
    "messages": [{"role": "user", "content": "Analyze the structure of this project. What are the main directories and what do they contain?"}]
}, config)
```

Good luck! 🚀

## 10. Summary: Model Context Protocol

**Key Takeaways:**

1. **MCP is a standardized protocol** for connecting AI agents to external services
   - Provides a common interface across different providers
   - Enables secure, efficient communication

2. **Two main integration approaches:**
   - **Direct API integration**: Simple, direct control (what we used with Tavily SDK)
   - **MCP protocol**: Standardized, better for multiple services

3. **MCP enables real-time capabilities:**
   - Web search (Tavily)
   - Database access
   - API integrations
   - File system operations
   - And much more!

4. **Security is critical:**
   - Never hardcode API keys
   - Use environment variables
   - Rotate keys regularly
   - Monitor usage

5. **Integration pattern:**
   ```python
   # 1. Get API key from provider
   api_key = os.getenv("PROVIDER_API_KEY")
   
   # 2. Create tool (direct API or MCP)
   tool = create_tool(api_key)
   
   # 3. Add to agent
   agent = ToolAgent(llm, tools=[tool])
   
   # 4. Use the agent
   response = agent.run("query")
   ```

6. **MCP vs Custom Tools:**
   - Custom tools: For your own code, simple needs
   - MCP: For external services, standardized integration, production systems

**Next Steps:**
- Explore other MCP servers (databases, file systems, etc.)
- Build agents that combine multiple MCP services
- Learn about local MCP server setup for production
- Practice error handling and security best practices

**Resources:**
- MCP Documentation: https://modelcontextprotocol.io
- LangChain Tools: https://python.langchain.com/docs/modules/tools/
- https://docs.langchain.com/oss/python/langchain/mcp
- https://docs.langchain.com/langsmith/server-mcp
- https://huggingface.co/learn/mcp-course/unit3/build-mcp-server
- https://github.com/daveebbelaar/ai-cookbook/tree/main/mcp/crash-course/3-simple-server-setup
- https://gist.github.com/teone/02b626d2faf5b1d2010888d857497eaa
- https://github.com/makenotion/notion-mcp-server?tab=readme-ov-file
- https://docs.tavily.com/documentation/mcp